# Online Retail: Customer Lifetime Value & Segmentation Analysis
**Author:** Thuy Nguyen  
**Dataset:** [Online Retail II — Kaggle](https://www.kaggle.com/datasets/jillwang87/online-retail-ii)  
**Tools:** Python (Pandas, NumPy, Matplotlib, Seaborn)

---

## Project Overview
This project analyzes 1M+ transactions from a UK-based online giftware retailer (Dec 2009 – Dec 2011).  
The goal is to move beyond "what happened" reporting to understanding **customer behavior** and driving **actionable retention strategies**.

**Business Questions:**
1. Which customer segments generate the most revenue?
2. What is the "tipping point" for repeat purchases?
3. Which products drive cross-selling behavior across segments?

---

## Table of Contents
1. [Setup & Data Loading](#1-setup--data-loading)
2. [Data Understanding](#2-data-understanding)
3. [Data Validation & Issue Discovery](#3-data-validation--issue-discovery)
4. [Data Cleaning](#4-data-cleaning)
5. [Data Aggregation](#5-data-aggregation)
6. [Customer Segmentation](#6-customer-segmentation)
7. [Segment Analysis](#7-segment-analysis)
8. [Time Behavior Analysis](#8-time-behavior-analysis)
9. [Annual Trend Analysis](#9-annual-trend-analysis)
10. [Business Recommendations](#10-business-recommendations)


## 1. Setup & Data Loading

In [ ]:
import zipfile
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)
sns.set_theme(style='whitegrid', palette='viridis')


In [ ]:
# ── Load Data ──────────────────────────────────────────────────────────────
# Option A: Google Colab (unzip from Drive)
zip_file_path = '/content/drive/MyDrive/Colab Notebooks/AceProjectData.zip'
extraction_dir = '/content/extracted_data'
os.makedirs(extraction_dir, exist_ok=True)

with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall(extraction_dir)

# Load both yearly files and combine
data_09_10 = pd.read_excel(f'{extraction_dir}/Year 2009-2010.xlsx', dtype={'CustomerID': str})
data_10_11 = pd.read_excel(f'{extraction_dir}/Year 2010-2011.xlsx', dtype={'CustomerID': str})
data = pd.concat([data_09_10, data_10_11], ignore_index=True)

# Option B: Local path (uncomment if running locally)
# data = pd.read_csv('online_retail_II.csv', dtype={'Customer ID': str})
# data.rename(columns={'Customer ID': 'CustomerID', 'Price': 'UnitPrice'}, inplace=True)

print(f"Dataset loaded. Shape: {data.shape}")
data.head()


## 2. Data Understanding

**Grain:** One row = one line item in a transaction (InvoiceNo + StockCode).  
If a customer buys 5 different products in one visit, there will be 5 rows sharing the same InvoiceNo.

| Column | Description |
|--------|-------------|
| InvoiceNo | Unique transaction ID. Prefix 'C' = cancellation, 'A' = bad debt adjustment |
| StockCode | Product code |
| Description | Product name |
| Quantity | Units purchased (negative = return/adjustment) |
| InvoiceDate | Date and time of transaction |
| UnitPrice | Price per unit in GBP |
| CustomerID | Unique customer identifier (~22.7% missing = guest checkouts) |
| Country | Customer's country |


In [ ]:
# Schema and basic stats
print("=== Data Types ===")
print(data.dtypes)
print(f"\n=== Shape: {data.shape} ===")
print("\n=== Descriptive Statistics ===")
data.describe()


In [ ]:
# Missing values analysis
missing_values = data.isnull().sum()
pct_missing = (missing_values / len(data)) * 100

missing_info = pd.DataFrame({
    'Missing Count': missing_values,
    'Percentage (%)': pct_missing
}).query('`Missing Count` > 0').sort_values('Percentage (%)', ascending=False)

print("Missing Values Summary:")
display(missing_info)
print("\nNote: ~22.7% of CustomerIDs are missing (guest checkouts). ")
print("These will be used for product trend analysis but excluded from customer segmentation.")


## 3. Data Validation & Issue Discovery

Systematic audit of all anomalous transaction types before cleaning.


### 3.1 Invoice Type Classification

In [ ]:
# Classify all invoices into types: Normal, Cancellation (C-prefix), Other (A-prefix)
def get_invoice_type(invoice_no_str):
    if invoice_no_str.startswith('C'):
        return 'Cancellation'
    try:
        int(invoice_no_str)
        return 'Normal'
    except ValueError:
        return 'Other (A-invoice / Bad Debt)'

data['InvoiceType'] = data['InvoiceNo'].astype(str).apply(get_invoice_type)
invoice_type_counts = data['InvoiceType'].value_counts()

print("Invoice Type Breakdown:")
display(invoice_type_counts)
print(f"\nCancellation rate: {invoice_type_counts.get('Cancellation', 0) / data['InvoiceNo'].nunique() * 100:.1f}% of unique invoices")

# Drop temp column
data = data.drop(columns=['InvoiceType'])


### 3.2 Quantity Anomalies

In [ ]:
# Overall quantity sign breakdown
pos_qty = (data['Quantity'] > 0).sum()
neg_qty = (data['Quantity'] < 0).sum()
total = len(data)

print(f"Positive quantities (sales):   {pos_qty:>10,} ({pos_qty/total*100:.2f}%)")
print(f"Negative quantities (returns): {neg_qty:>10,} ({neg_qty/total*100:.2f}%)")

# Key finding: C-invoices should always have negative quantities
cancellation_rows = data[data['InvoiceNo'].astype(str).str.startswith('C', na=False)]
positive_qty_cancellations = cancellation_rows[cancellation_rows['Quantity'] >= 0]
print(f"\nC-invoices with positive quantity (anomaly): {len(positive_qty_cancellations)} rows")
if not positive_qty_cancellations.empty:
    display(positive_qty_cancellations)


In [ ]:
# Numeric invoices with negative quantities (no C or A prefix) — unexpected
numeric_negatives = data[
    (data['Quantity'] < 0) &
    (~data['InvoiceNo'].astype(str).str.startswith(('C', 'A'), na=False))
]

print(f"Numeric invoices with negative quantities: {len(numeric_negatives):,} rows")
print("\nUnitPrice breakdown for these rows:")
print(f"  Positive UnitPrice: {(numeric_negatives['UnitPrice'] > 0).sum()}")
print(f"  Zero UnitPrice:     {(numeric_negatives['UnitPrice'] == 0).sum()}")
print(f"  Negative UnitPrice: {(numeric_negatives['UnitPrice'] < 0).sum()}")
print("\n→ All 3,457 rows have UnitPrice = $0.00 → store corrections, not customer returns.")
print("\nSample rows:")
display(numeric_negatives.head(5))


### 3.3 UnitPrice Anomalies

In [ ]:
# UnitPrice distribution
zero_price = (data['UnitPrice'] == 0).sum()
neg_price  = (data['UnitPrice'] < 0).sum()
pos_price  = (data['UnitPrice'] > 0).sum()
total = len(data)

print(f"Positive UnitPrice: {pos_price:>10,} ({pos_price/total*100:.2f}%)")
print(f"Zero UnitPrice:     {zero_price:>10,} ({zero_price/total*100:.2f}%) — samples/promos/data errors")
print(f"Negative UnitPrice: {neg_price:>10,} ({neg_price/total*100:.2f}%) — bad debt adjustments (A-invoices)")
print(f"\nUnitPrice range: {data['UnitPrice'].min()} to {data['UnitPrice'].max()}")

# Top descriptions for zero-price items
print("\nTop 10 descriptions for zero-price items:")
display(data[data['UnitPrice'] == 0]['Description'].value_counts().head(10))


### 3.4 Duplicate Analysis

In [ ]:
# Primary Key analysis: InvoiceNo + StockCode
pk_cols = ['InvoiceNo', 'StockCode']
pk_dups = data.duplicated(subset=pk_cols).sum()
exact_dups = data.duplicated().sum()
total = len(data)

print("=== Primary Key Analysis (InvoiceNo + StockCode) ===")
print(f"Total rows:                         {total:>10,}")
print(f"PK duplicates:                      {pk_dups:>10,} ({pk_dups/total*100:.2f}%)")
print(f"100% identical rows:                {exact_dups:>10,} ({exact_dups/total*100:.2f}%) → safe to remove")
print(f"Partial PK duplicates (diff values):{pk_dups - exact_dups:>10,} → tiered pricing / system logging → keep")
print("\nFinding: Partial duplicates likely represent tiered pricing or multi-line logging. Keep them.")


## 4. Data Cleaning

**Cleaning decisions summary:**

| Issue | Action | Reason |
|-------|--------|--------|
| 3,457 numeric invoices with negative qty + $0 price | Remove | Store corrections, not customer transactions |
| 6 'A-invoice' bad debt adjustments | Remove | Accounting entries, not sales |
| 1 C-invoice with positive quantity | Remove | Data anomaly |
| 34,335 exact duplicate rows | Remove one copy | Data entry/logging error |
| ~22.7% missing CustomerID | Keep | Guest checkouts — useful for product analysis |


In [ ]:
# Step 1: Remove the 3 targeted groups of anomalous rows
idx_numeric_neg = data[
    (data['Quantity'] < 0) &
    (~data['InvoiceNo'].astype(str).str.startswith(('C', 'A'), na=False))
].index

idx_a_invoices = data[data['InvoiceNo'].astype(str).str.startswith('A', na=False)].index

idx_c_pos_qty = data[
    (data['InvoiceNo'].astype(str).str.startswith('C', na=False)) &
    (data['Quantity'] > 0)
].index

indices_to_remove = idx_numeric_neg.union(idx_a_invoices).union(idx_c_pos_qty)
data_cleaned = data.drop(index=indices_to_remove).copy()

print(f"Removed {len(indices_to_remove):,} anomalous rows:")
print(f"  - Numeric negative qty (store corrections): {len(idx_numeric_neg):,}")
print(f"  - A-invoice bad debt adjustments:           {len(idx_a_invoices):,}")
print(f"  - C-invoice with positive quantity:         {len(idx_c_pos_qty):,}")
print(f"\nOriginal shape: {data.shape}")
print(f"Cleaned shape:  {data_cleaned.shape}")


In [ ]:
# Step 2: Remove exact duplicate rows (keep first occurrence)
before = len(data_cleaned)
data_cleaned = data_cleaned.drop_duplicates()
after = len(data_cleaned)
print(f"Removed {before - after:,} exact duplicate rows.")
print(f"Final cleaned shape: {data_cleaned.shape}")


In [ ]:
# Before vs After cleaning — business impact
data_cleaned['TotalRevenue'] = data_cleaned['Quantity'] * data_cleaned['UnitPrice']

before_stats = {
    'Total Revenue': (data['Quantity'] * data['UnitPrice']).sum(),
    'Unique Customers': data['CustomerID'].nunique(),
    'Unique Invoices': data['InvoiceNo'].nunique(),
    'Row Count': len(data)
}
after_stats = {
    'Total Revenue': data_cleaned['TotalRevenue'].sum(),
    'Unique Customers': data_cleaned['CustomerID'].nunique(),
    'Unique Invoices': data_cleaned['InvoiceNo'].nunique(),
    'Row Count': len(data_cleaned)
}

impact = pd.DataFrame({'Before Cleaning': before_stats, 'After Cleaning': after_stats})
impact['Change (%)'] = ((impact['After Cleaning'] - impact['Before Cleaning']) / impact['Before Cleaning'] * 100).round(2)
print("Business Impact of Cleaning:")
display(impact)


## 5. Data Aggregation

Building 3 analytical tables from the cleaned data:
- **Item level** — one row per product per invoice
- **Order level** — one row per invoice
- **Customer level** — one row per customer


In [ ]:
# ── Item Level Table ─────────────────────────────────────────────────────
# Aggregate duplicate (InvoiceNo, StockCode) pairs: sum qty, average price
item_agg = {
    'Quantity':    'sum',
    'UnitPrice':   'mean',
    'Description': 'first',
    'InvoiceDate': 'first',
    'CustomerID':  'first',
    'Country':     'first',
    'TotalRevenue':'sum'
}
data_item_level = (data_cleaned
    .groupby(['InvoiceNo', 'StockCode'], observed=True)
    .agg(item_agg)
    .reset_index()
)
data_item_level = data_item_level[data_item_level['Quantity'] != 0]
data_item_level['InvoiceDate'] = pd.to_datetime(data_item_level['InvoiceDate'])
data_item_level['InvoiceYear'] = data_item_level['InvoiceDate'].dt.year

print(f"Item level table shape: {data_item_level.shape}")
display(data_item_level.head(3))


In [ ]:
# ── Order Level Table ────────────────────────────────────────────────────
order_agg = {
    'TotalRevenue': 'sum',
    'Quantity':     'sum',
    'StockCode':    'nunique',
    'InvoiceDate':  'first',
    'CustomerID':   'first',
    'Country':      'first'
}
data_orders = (data_item_level
    .groupby('InvoiceNo', observed=True)
    .agg(order_agg)
    .reset_index()
    .rename(columns={'Quantity': 'TotalQuantity', 'StockCode': 'UniqueProducts'})
)
data_orders['IsCancelled'] = data_orders['InvoiceNo'].astype(str).str.startswith('C')
data_orders['InvoiceDate'] = pd.to_datetime(data_orders['InvoiceDate'])

print(f"Order level table shape: {data_orders.shape}")
display(data_orders.head(3))


In [ ]:
# ── Customer Level Table ─────────────────────────────────────────────────
customer_agg = {
    'InvoiceNo':    'nunique',
    'InvoiceDate':  'max',
    'TotalRevenue': 'sum',
    'IsCancelled':  'mean'
}
data_customers = (data_orders
    .groupby('CustomerID', observed=True)
    .agg(customer_agg)
    .reset_index()
    .rename(columns={
        'InvoiceNo':    'TotalInvoices',
        'InvoiceDate':  'LastPurchaseDate',
        'TotalRevenue': 'TotalCustomerRevenue',
        'IsCancelled':  'CancellationRate'
    })
)
data_customers['LastPurchaseDate'] = pd.to_datetime(data_customers['LastPurchaseDate'])
max_date = data_customers['LastPurchaseDate'].max()
data_customers['Recency'] = (max_date - data_customers['LastPurchaseDate']).dt.days

print(f"Customer level table shape: {data_customers.shape}")
display(data_customers.head(3))
print("\nDescriptive stats:")
display(data_customers[['TotalInvoices', 'TotalCustomerRevenue', 'CancellationRate', 'Recency']].describe())


## 6. Customer Segmentation

**Segmentation logic (priority order):**

| Priority | Segment | Definition |
|----------|---------|------------|
| 1 | Risky | CancellationRate > 50% AND TotalRevenue > 0 |
| 2 | High Value | Top 20% revenue AND top 20% frequency AND CancellationRate ≤ 50% |
| 3 | Frequent but Low Spend | Top 20% frequency (≥10 invoices) but below revenue threshold |
| 4 | One-time Buyer | Exactly 1 invoice AND CancellationRate = 0 |
| 5 | Low Value | >1 purchase, below high revenue and high frequency thresholds |
| 6 | Other | Remaining edge cases |


In [ ]:
# Define thresholds
rev_threshold_high  = data_customers['TotalCustomerRevenue'].quantile(0.80)
freq_threshold_high = data_customers['TotalInvoices'].quantile(0.80)
cancel_threshold    = 0.50

print(f"Revenue threshold (top 20%):   £{rev_threshold_high:,.2f}")
print(f"Frequency threshold (top 20%): {freq_threshold_high:.0f} invoices")
print(f"Cancellation threshold:        {cancel_threshold*100:.0f}%")

def segment_customer(row):
    if row['CancellationRate'] > cancel_threshold and row['TotalCustomerRevenue'] > 0:
        return 'Risky'
    if (row['TotalCustomerRevenue'] >= rev_threshold_high and
        row['CancellationRate'] <= cancel_threshold and
        row['TotalInvoices'] >= freq_threshold_high):
        return 'High Value'
    if row['TotalInvoices'] >= freq_threshold_high:
        return 'Frequent but Low Spend'
    if row['TotalInvoices'] == 1 and row['CancellationRate'] == 0:
        return 'One-time Buyer'
    if (row['TotalInvoices'] > 1 and
        row['TotalCustomerRevenue'] < rev_threshold_high and
        row['TotalInvoices'] < freq_threshold_high):
        return 'Low Value'
    return 'Other'

data_customers['Segment'] = data_customers.apply(segment_customer, axis=1)

# Coverage check
total = len(data_customers)
other_count = (data_customers['Segment'] == 'Other').sum()
print(f"\nSegment coverage: {(total - other_count)/total*100:.2f}%")
print("\nSegment counts:")
display(data_customers['Segment'].value_counts())


## 7. Segment Analysis

In [ ]:
# Segment comparison table
total_customers = len(data_customers)
total_revenue   = data_customers['TotalCustomerRevenue'].sum()

comparison_table = (data_customers
    .groupby('Segment', observed=True)
    .agg(
        CustomerCount=('CustomerID', 'count'),
        SegmentRevenue=('TotalCustomerRevenue', 'sum'),
        AvgRevenue=('TotalCustomerRevenue', 'mean'),
        AvgInvoices=('TotalInvoices', 'mean'),
        AvgCancellationRate=('CancellationRate', 'mean')
    )
    .reset_index()
)
comparison_table['% of Customers'] = (comparison_table['CustomerCount'] / total_customers * 100).round(2)
comparison_table['% of Revenue']   = (comparison_table['SegmentRevenue']  / total_revenue  * 100).round(2)
comparison_table['AvgRevenue']      = comparison_table['AvgRevenue'].round(2)
comparison_table['AvgInvoices']     = comparison_table['AvgInvoices'].round(1)
comparison_table['AvgCancellationRate'] = (comparison_table['AvgCancellationRate'] * 100).round(1)

display_cols = ['Segment', '% of Customers', '% of Revenue', 'AvgRevenue', 'AvgInvoices', 'AvgCancellationRate']
display(comparison_table[display_cols].sort_values('% of Revenue', ascending=False))


In [ ]:
# Avg revenue per invoice per segment
segment_invoice_revenue = (data_customers
    .merge(data_orders[['CustomerID', 'InvoiceNo', 'TotalRevenue']], on='CustomerID', how='left')
    .groupby('Segment', observed=True)
    .agg(
        TotalSegmentRevenue=('TotalRevenue', 'sum'),
        TotalInvoices=('InvoiceNo', 'nunique')
    )
    .reset_index()
)
segment_invoice_revenue['AvgRevenuePerInvoice'] = (
    segment_invoice_revenue['TotalSegmentRevenue'] / segment_invoice_revenue['TotalInvoices']
).round(2)

print("Average Revenue per Invoice by Segment:")
display(segment_invoice_revenue[['Segment', 'AvgRevenuePerInvoice']].sort_values('AvgRevenuePerInvoice', ascending=False))


In [ ]:
# Revenue distribution bar chart
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

seg_order = comparison_table.sort_values('% of Revenue', ascending=False)['Segment']

axes[0].bar(seg_order, comparison_table.set_index('Segment').loc[seg_order, '% of Revenue'], color=sns.color_palette('viridis', len(seg_order)))
axes[0].set_title('% of Total Revenue by Segment')
axes[0].set_ylabel('% of Revenue')
axes[0].set_xticklabels(seg_order, rotation=30, ha='right')

axes[1].bar(seg_order, comparison_table.set_index('Segment').loc[seg_order, '% of Customers'], color=sns.color_palette('magma', len(seg_order)))
axes[1].set_title('% of Total Customers by Segment')
axes[1].set_ylabel('% of Customers')
axes[1].set_xticklabels(seg_order, rotation=30, ha='right')

plt.tight_layout()
plt.show()
print("Key insight: High Value customers (~19.9% of base) drive ~77% of total revenue.")


## 8. Time Behavior Analysis

In [ ]:
# Merge orders with segments, extract time features
data_orders['InvoiceDate'] = pd.to_datetime(data_orders['InvoiceDate'])
data_orders['DayOfWeek']   = data_orders['InvoiceDate'].dt.day_name()
data_orders['HourOfDay']   = data_orders['InvoiceDate'].dt.hour
data_orders['Month']       = data_orders['InvoiceDate'].dt.month_name()

segmented_orders = data_orders.merge(
    data_customers[['CustomerID', 'Segment']], on='CustomerID', how='left'
)
segmented_orders['Segment'] = segmented_orders['Segment'].fillna('Unknown')


In [ ]:
# Day of week
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

plt.figure(figsize=(14, 6))
sns.countplot(data=segmented_orders, x='DayOfWeek', hue='Segment', order=day_order, palette='viridis')
plt.title('Shopping Frequency by Day of Week — by Segment')
plt.xlabel('Day of Week')
plt.ylabel('Number of Orders')
plt.xticks(rotation=30)
plt.legend(title='Segment', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()
print("Insight: All segments peak mid-week (Tue–Thu). Weekend activity drops significantly.")


In [ ]:
# Hour of day
plt.figure(figsize=(14, 6))
sns.countplot(data=segmented_orders, x='HourOfDay', hue='Segment', palette='magma')
plt.title('Shopping Frequency by Hour of Day — by Segment')
plt.xlabel('Hour (24h)')
plt.ylabel('Number of Orders')
plt.xticks(range(0, 24))
plt.legend(title='Segment', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()
print("Insight: Peak shopping hours are 10 AM – 3 PM across all segments.")


In [ ]:
# Month
month_order = ['January','February','March','April','May','June',
               'July','August','September','October','November','December']

plt.figure(figsize=(16, 6))
sns.countplot(data=segmented_orders, x='Month', hue='Segment', order=month_order, palette='cividis')
plt.title('Shopping Frequency by Month — by Segment')
plt.xlabel('Month')
plt.ylabel('Number of Orders')
plt.xticks(rotation=30)
plt.legend(title='Segment', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()
print("Insight: November is the dominant peak month (holiday gifting season). Jan–Feb are the slowest.")


## 9. Annual Trend Analysis

In [ ]:
# Build yearly customer table (re-segment per year)
data_item_level['InvoiceDate'] = pd.to_datetime(data_item_level['InvoiceDate'])
data_item_level['InvoiceYear'] = data_item_level['InvoiceDate'].dt.year

orders_for_merge = data_orders[['InvoiceNo', 'IsCancelled']]
data_item_merged = data_item_level.merge(orders_for_merge, on='InvoiceNo', how='left')

data_customers_yearly = (data_item_merged
    .groupby(['CustomerID', 'InvoiceYear'], observed=True)
    .agg(
        TotalInvoices=('InvoiceNo', 'nunique'),
        LastPurchaseDate=('InvoiceDate', 'max'),
        TotalCustomerRevenue=('TotalRevenue', 'sum'),
        CancellationRate=('IsCancelled', 'mean')
    )
    .reset_index()
)

# Re-apply segmentation per year
data_customers_yearly['LastPurchaseDate'] = pd.to_datetime(data_customers_yearly['LastPurchaseDate'])
max_date_y = data_customers_yearly['LastPurchaseDate'].max()
data_customers_yearly['Recency'] = (max_date_y - data_customers_yearly['LastPurchaseDate']).dt.days

rev_y  = data_customers_yearly['TotalCustomerRevenue'].quantile(0.80)
freq_y = data_customers_yearly['TotalInvoices'].quantile(0.80)

data_customers_yearly['Segment'] = data_customers_yearly.apply(segment_customer, axis=1)
print(f"Yearly customer table shape: {data_customers_yearly.shape}")
display(data_customers_yearly.head(3))


In [ ]:
# Annual segment performance
segment_annual = (data_customers_yearly
    .groupby(['InvoiceYear', 'Segment'], observed=True)
    .agg(
        CustomerCount=('CustomerID', 'nunique'),
        AnnualRevenue=('TotalCustomerRevenue', 'sum')
    )
    .reset_index()
)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.lineplot(data=segment_annual, x='InvoiceYear', y='CustomerCount',
             hue='Segment', marker='o', ax=axes[0])
axes[0].set_title('Customer Count by Segment Over Years')
axes[0].set_xlabel('Year')
axes[0].set_ylabel('Customer Count')
axes[0].grid(True)
axes[0].legend(title='Segment', bbox_to_anchor=(1.05, 1), loc='upper left')

sns.lineplot(data=segment_annual, x='InvoiceYear', y='AnnualRevenue',
             hue='Segment', marker='o', ax=axes[1])
axes[1].set_title('Annual Revenue by Segment Over Years')
axes[1].set_xlabel('Year')
axes[1].set_ylabel('Revenue (£)')
axes[1].grid(True)
axes[1].legend(title='Segment', bbox_to_anchor=(1.05, 1), loc='upper left')

plt.tight_layout()
plt.show()


## 10. Business Recommendations

| Segment | % Customers | % Revenue | Strategy |
|---------|-------------|-----------|---------|
| **High Value** | ~19.9% | ~77.1% | Loyalty programs, VIP early access, cross-sell premium products |
| **Low Value** | ~49.3% | ~16.4% | Increase AOV via bundles and free-shipping thresholds |
| **One-time Buyer** | ~23.5% | ~2.5% | Post-purchase follow-up email, second-purchase discount |
| **Frequent but Low Spend** | ~5.2% | ~3.7% | Upsell to premium alternatives, subscription offers |
| **Risky** | ~1.2% | ~0.6% | Investigate cancellation root causes, improve product descriptions |

**Key Takeaways:**
- High Value customers (~20%) generate ~77% of revenue → retention is the #1 priority
- One-time buyers (~24%) represent the largest growth opportunity if conversion rate improves
- November is the peak month — plan inventory and campaigns 2 months in advance
- Shopping peaks mid-week (Tue–Thu), 10 AM–3 PM → optimal email/promotion send times
